In [ ]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)
import torch
import torch.nn as nn
import torch.optim as optim
import copy, time, random
from numpy.lib.stride_tricks import sliding_window_view

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Using: cpu


In [ ]:
unsw_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "dhoogla/unswnb15", "UNSW_NB15_training-set.parquet"
)
cicids_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "chethuhn/network-intrusion-dataset", "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

Using Colab cache for faster access to the 'unswnb15' dataset.
Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.


In [ ]:
# Strip hidden trailing whitespaces from column headers
unsw_source.columns = unsw_source.columns.str.strip()
cicids_source.columns = cicids_source.columns.str.strip()

# Same label column name in both datasets
unsw_source.rename(columns={'attack_cat': 'attack_type'}, inplace=True)
cicids_source.rename(columns={'Label': 'attack_type'}, inplace=True)

# Normal -> Benign
unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()
cicids_source['attack_type'] = cicids_source['attack_type'].replace('BENIGN', 'Benign').str.strip()

/tmp/ipykernel_2366/2438950036.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()


In [ ]:
print("-> Extracting and aligning 13 core network features...")
unsw_cols_map = [
    'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'sload', 'dload',
    'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'rate'
]

df_unsw = unsw_source[unsw_cols_map].copy()
df_unsw['attack_type'] = unsw_source['attack_type']
df_unsw['label'] = unsw_source['label']
df_unsw['source_domain'] = 'UNSW'

cicids_corrected_mapping = {
    'Flow Duration': 'dur', 'Total Fwd Packets': 'spkts', 'Total Backward Packets': 'dpkts',
    'Total Length of Fwd Packets': 'sbytes', 'Total Length of Bwd Packets': 'dbytes',
    'Flow Bytes/s': 'sload', 'Fwd Packets/s': 'dload', 'Flow IAT Std': 'sjit',
    'Bwd IAT Std': 'djit', 'Flow IAT Mean': 'tcprtt', 'Fwd IAT Mean': 'synack',
    'Bwd IAT Mean': 'ackdat', 'Flow Packets/s': 'rate'
}

df_cicids = cicids_source[list(cicids_corrected_mapping.keys())].copy()
df_cicids.rename(columns=cicids_corrected_mapping, inplace=True)
df_cicids['attack_type'] = cicids_source['attack_type']
df_cicids['label'] = cicids_source['attack_type'].apply(lambda x: 0 if x == 'Benign' else 1)
df_cicids['source_domain'] = 'CICIDS'

-> Extracting and aligning 13 core network features...


In [ ]:
# inf / NaN removal (index = original row number is KEPT, we need it later for ordering)
for df_temp in [df_unsw, df_cicids]:
    df_temp.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
    df_temp.dropna(inplace=True)

# MinMax scaling (same as baseline)
scaler = MinMaxScaler()
df_unsw[unsw_cols_map] = scaler.fit_transform(df_unsw[unsw_cols_map].astype(np.float32))
df_cicids[unsw_cols_map] = scaler.fit_transform(df_cicids[unsw_cols_map].astype(np.float32))

In [ ]:
partitions = {}

unsw_b_master = df_unsw[df_unsw['attack_type'] == 'Benign'].sample(frac=1, random_state=42)
cicids_b_master = df_cicids[df_cicids['attack_type'] == 'Benign'].sample(frac=1, random_state=42)

unsw_splits = np.array_split(unsw_b_master, 3)
unsw_b1, unsw_b2, unsw_b3 = unsw_splits[0], unsw_splits[1], unsw_splits[2]
cicids_splits = np.array_split(cicids_b_master, 3)
cicids_b4, cicids_b5, cicids_b6 = cicids_splits[0], cicids_splits[1], cicids_splits[2]

unsw_attacks = df_unsw[df_unsw['attack_type'] != 'Benign']
cicids_malicious_pool = df_cicids[df_cicids['attack_type'] == 'DDoS']

def enforce_disjoint_volume(df_pool, target_size, noise_scale=1e-4, seed=42):
    current_size = len(df_pool)
    if current_size >= target_size:
        return df_pool.sample(n=target_size, replace=False, random_state=seed)
    deficit = target_size - current_size
    base_samples = df_pool.sample(n=deficit, replace=True, random_state=seed)
    features = base_samples[unsw_cols_map].values
    np.random.seed(seed)
    noise = np.random.normal(0, noise_scale, features.shape)
    perturbed_features = np.clip(features + noise, 0.0, 1.0)
    df_synthetic = pd.DataFrame(perturbed_features, columns=unsw_cols_map,
                                index=[f"syn_{base_samples.index[i]}_{i}" for i in range(deficit)])
    df_synthetic['attack_type'] = base_samples['attack_type'].values
    df_synthetic['label'] = base_samples['label'].values
    df_synthetic['source_domain'] = base_samples['source_domain'].values
    return pd.concat([df_pool, df_synthetic])

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
partitions['D1'] = pd.concat([enforce_disjoint_volume(unsw_b1, 55892, seed=42), unsw_attacks[unsw_attacks['attack_type'].isin(['Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms'])].sample(n=17381, replace=True, random_state=42)])
partitions['D2'] = pd.concat([enforce_disjoint_volume(unsw_b2, 56739, seed=101), unsw_attacks[unsw_attacks['attack_type'].isin(['DoS', 'Backdoor'])].sample(n=15972, replace=True, random_state=101)])
partitions['D3'] = pd.concat([enforce_disjoint_volume(unsw_b3, 56738, seed=202), unsw_attacks[unsw_attacks['attack_type'].isin(['Exploits', 'Analysis'])].sample(n=109916, replace=True, random_state=202)])
partitions['D4'] = pd.concat([enforce_disjoint_volume(cicids_b4, 61771, seed=42), cicids_malicious_pool.sample(n=54528, replace=True, random_state=42)])
partitions['D5'] = pd.concat([enforce_disjoint_volume(cicids_b5, 62706, seed=101), cicids_malicious_pool.sample(n=30497, replace=True, random_state=101)])
partitions['D6'] = pd.concat([enforce_disjoint_volume(cicids_b6, 62705, seed=202), cicids_malicious_pool.sample(n=53342, replace=True, random_state=202)])

# Stratified cross-network validation pool (same as baseline)
_, unsw_eval_slice = train_test_split(df_unsw, test_size=0.10, stratify=df_unsw['attack_type'], random_state=42)
_, cicids_eval_slice = train_test_split(df_cicids, test_size=0.10, stratify=df_cicids['attack_type'], random_state=42)
master_eval_pool = pd.concat([unsw_eval_slice, cicids_eval_slice])
df_eval = master_eval_pool.sample(n=35451, replace=True, random_state=42)   # baseline eval (has leakage + duplicates)

In [ ]:
research_paper_blueprint = {
    'D1': {'Total': 73183, 'Source': 'UNSW', 'Counts': {'Benign': 55892, 'Fuzzers': 12973, 'Generic': 3415, 'Reconnaissance': 795, 'Shellcode': 76, 'Worms': 32}},
    'D2': {'Total': 72711, 'Source': 'UNSW', 'Counts': {'Benign': 56739, 'DoS': 15927, 'Backdoor': 45}},
    'D3': {'Total': 166654, 'Source': 'UNSW', 'Counts': {'Benign': 56738, 'Exploits': 109862, 'Analysis': 54}},
    'D4': {'Total': 116299, 'Source': 'CICIDS', 'Counts': {'Benign': 61771, 'DDoS': 54528}},
    'D5': {'Total': 93203, 'Source': 'CICIDS', 'Counts': {'Benign': 62706, 'DDoS': 30497}},
    'D6': {'Total': 116047, 'Source': 'CICIDS', 'Counts': {'Benign': 62705, 'DDoS': 53342}}
}

for client_id, meta in research_paper_blueprint.items():
    client_df = partitions[client_id]
    actual_total, expected_total = len(client_df), meta['Total']
    print(f"{client_id} ({meta['Source']}) | actual={actual_total} expected={expected_total} "
          f"{'MATCHED' if actual_total == expected_total else 'DIFF'}")
    actual_counts = client_df['attack_type'].value_counts()
    for attack_cls, exp_count in meta['Counts'].items():
        act = actual_counts.get(attack_cls, 0)
        print(f"    {attack_cls:<16} actual={act:<7} expected={exp_count:<7} {'MATCHED' if act == exp_count else 'DIFF'}")
print("Eval (baseline) rows:", len(df_eval))

D1 (UNSW) | actual=73273 expected=73183 DIFF
    Benign           actual=55892   expected=55892   MATCHED
    Fuzzers          actual=4591    expected=12973   DIFF
    Generic          actual=9789    expected=3415    DIFF
    Reconnaissance   actual=2683    expected=795     DIFF
    Shellcode        actual=285     expected=76      DIFF
    Worms            actual=33      expected=32      DIFF
D2 (UNSW) | actual=72711 expected=72711 MATCHED
    Benign           actual=56739   expected=56739   MATCHED
    DoS              actual=13931   expected=15927   DIFF
    Backdoor         actual=2041    expected=45      DIFF
D3 (UNSW) | actual=166654 expected=166654 MATCHED
    Benign           actual=56738   expected=56738   MATCHED
    Exploits         actual=103778  expected=109862  DIFF
    Analysis         actual=6138    expected=54      DIFF
D4 (CICIDS) | actual=116299 expected=116299 MATCHED
    Benign           actual=61771   expected=61771   MATCHED
    DDoS             actual=54528   exp

In [ ]:
DOM = {'UNSW': 0, 'CICIDS': 1}

def orig_id(df):
    # normal row: index = original row number; synthetic row 'syn_<orig>_<i>' -> <orig>
    return np.array([int(str(i).split('_')[1]) if isinstance(i, str) else int(i) for i in df.index])

def row_key(df):
    # unique id = (dataset, original row number). Key sorting = file order (UNSW first, then CICIDS)
    return df['source_domain'].map(DOM).values.astype(np.int64) * 10**9 + orig_id(df)

# Eval: unique 10% slice (no replace=True, no duplicates)
df_eval_clean = master_eval_pool.copy()
df_eval_clean['_key'] = row_key(df_eval_clean)
assert df_eval_clean['_key'].is_unique
eval_keys = df_eval_clean['_key'].values

# Partitions copy without any eval row
clean_partitions = {}
for cid, df in partitions.items():
    df = df.copy()
    df['_key'] = row_key(df)
    clean_partitions[cid] = df[~np.isin(df['_key'].values, eval_keys)]

for cid, df in clean_partitions.items():
    print(cid, "rows:", len(df), "| overlap with eval:", int(np.isin(df['_key'].values, eval_keys).sum()))
print("Eval rows:", len(df_eval_clean), "| attack share:", round(df_eval_clean['label'].mean(), 3))

D1 rows: 70852 | overlap with eval: 0
D2 rows: 70413 | overlap with eval: 0
D3 rows: 139935 | overlap with eval: 0
D4 rows: 110894 | overlap with eval: 0
D5 rows: 90223 | overlap with eval: 0
D6 rows: 91810 | overlap with eval: 0
Eval rows: 40107 | attack share: 0.617


In [ ]:
# Does the file order carry any temporal structure? (decides if windows make sense)
def adjacency_check(df, name):
    y = df.sort_index()['label'].values
    same_adj = (y[1:] == y[:-1]).mean()
    p = y.mean()
    print(f"{name}: adjacent same-label = {same_adj:.3f} | random expectation = {p**2 + (1-p)**2:.3f}")

adjacency_check(df_unsw, "UNSW   (file order)")
adjacency_check(df_cicids, "CICIDS (file order)")
# same_adj ~ random  -> rows are shuffled, windows add no temporal info
# same_adj >> random -> file has order (bursts), windows can help

UNSW   (file order): adjacent same-label = 0.953 | random expectation = 0.565
CICIDS (file order): adjacent same-label = 1.000 | random expectation = 0.509


In [ ]:
class FcNN_Baseline(nn.Module):
    def __init__(self, input_dim=13):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 4), nn.ReLU(), nn.Linear(4, 2), nn.ReLU(),
            nn.Linear(2, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)

class OneD_CNN_Baseline(nn.Module):
    def __init__(self, input_dim=13):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv1d(1, 16, kernel_size=2), nn.ReLU())
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(nn.Linear(12 * 16, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        if len(x.shape) == 2: x = x.unsqueeze(1)
        return self.fc(self.flatten(self.conv1(x)))

class GRU_IDS(nn.Module):
    def __init__(self, input_size=13, hidden=32):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):                 # x: (B, T, 13)
        _, h = self.rnn(x)                # h: (1, B, hidden) = summary of the whole window
        return self.head(h[-1])

class LSTM_IDS(nn.Module):
    def __init__(self, input_size=13, hidden=32):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        _, (h, c) = self.rnn(x)
        return self.head(h[-1])

MODEL_MODE = {"FcNN": "tabular", "OneD_CNN": "tabular", "GRU": "window", "LSTM": "window"}

def build_model(model_type):
    return {"FcNN": FcNN_Baseline, "OneD_CNN": OneD_CNN_Baseline,
            "GRU": GRU_IDS, "LSTM": LSTM_IDS}[model_type]()

In [ ]:
def prepare_xy(df, mode="tabular", window=10, dedup=False):
    if dedup:
        df = df.drop_duplicates('_key')
    if mode == "window":
        df = df.sort_values('_key', kind='stable')     # time/file order, NOT random
    X = df[unsw_cols_map].values.astype(np.float32)
    y = df['label'].values.astype(np.float32)
    if mode == "tabular":
        return X, y
    if mode == "window":
        Xw = np.ascontiguousarray(sliding_window_view(X, (window, X.shape[1]))[:, 0])  # (N-T+1, T, 13)
        return Xw, y[window - 1:]                       # label of the LAST flow in window
    raise ValueError(mode)

In [ ]:
def client_update(model, train_df, local_epochs, lr, batch_size, mode="tabular", window=10, dedup=False):
    model.to(DEVICE).train()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    X, y = prepare_xy(train_df, mode, window, dedup)
    X = torch.from_numpy(X).to(DEVICE)
    y = torch.from_numpy(y).unsqueeze(1).to(DEVICE)
    n = len(X)

    for _ in range(local_epochs):
        perm = torch.randperm(n, device=DEVICE)         # shuffles WINDOWS (samples), never rows inside a window
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = criterion(model(X[idx]), y[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # same for all models
            optimizer.step()
    flat = torch.cat([p.data.view(-1) for p in model.parameters()]).cpu()
    return flat, n

In [ ]:
class AdaptiveServerOptimizer:
    """ FedAvg / FedAdam / FedYogi (same as baseline) """
    def __init__(self, model_shape, strategy="FedAvg", beta1=0.9, beta2=0.99, eta=0.1, tau=1e-3):
        self.strategy, self.beta1, self.beta2, self.eta, self.tau = strategy, beta1, beta2, eta, tau
        self.m_t = torch.zeros(model_shape)
        self.v_t = torch.ones(model_shape) * (tau ** 2)

    def aggregate(self, global_weights, client_updates_list, client_sizes):
        flat_global = torch.cat([p.data.view(-1) for p in global_weights.parameters()])

        if self.strategy == "FedAvg":
            total_n = sum(client_sizes)
            avg_update = torch.zeros_like(flat_global)
            for idx, c_weights in enumerate(client_updates_list):
                avg_update += (client_sizes[idx] / total_n) * c_weights
            return avg_update
        else:
            delta_t = torch.stack([c_w - flat_global for c_w in client_updates_list]).mean(dim=0)
            self.m_t = self.beta1 * self.m_t + (1.0 - self.beta1) * delta_t
            delta_sq = delta_t ** 2
            if self.strategy == "FedYogi":
                self.v_t = self.v_t - (1.0 - self.beta2) * delta_sq * torch.sign(self.v_t - delta_sq)
            elif self.strategy == "FedAdam":
                self.v_t = self.beta2 * self.v_t + (1.0 - self.beta2) * delta_sq
            return flat_global + self.eta * (self.m_t / (torch.sqrt(self.v_t) + self.tau))

def unpack_weights(model, flat_params):
    ptr = 0
    with torch.no_grad():
        for p in model.parameters():
            n = p.numel()
            p.copy_(flat_params[ptr:ptr + n].view_as(p))
            ptr += n

In [ ]:
def evaluate_global_model(model, eval_df, mode="tabular", window=10):
    m = copy.deepcopy(model).to(DEVICE).eval()
    X, y = prepare_xy(eval_df, mode, window)            # window mode: eval is also sorted by file order
    Xt = torch.from_numpy(X).to(DEVICE)
    with torch.no_grad():
        p = torch.cat([m(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)]).cpu().numpy().ravel()
    yhat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, yhat), f1=f1_score(y, yhat, zero_division=0),
                prec=precision_score(y, yhat, zero_division=0),
                rec=recall_score(y, yhat, zero_division=0),
                auc=roc_auc_score(y, p), fpr=fp / max(fp + tn, 1))

In [ ]:
def execute_federated_experiment(partitions_pool, eval_set, model_type="FcNN", strategy="FedAvg",
                                 local_epochs=5, lr=0.001, window=10, seed=42, episodes=5,
                                 batch_size=64, dedup=None):
    mode = MODEL_MODE[model_type]
    if dedup is None:
        dedup = (mode == "window")
    ordered = (mode == "window") or dedup               # ordered/clean mode: no bootstrap
    print(f"\n[{model_type}] + [{strategy}] | epochs={local_epochs} | mode={mode} | dedup={dedup}"
          + (f" | window={window}" if mode == "window" else ""))

    set_seed(seed)
    global_model = build_model(model_type)              # stays on CPU; clients train on a GPU copy
    total_params = sum(p.numel() for p in global_model.parameters())
    server = AdaptiveServerOptimizer(model_shape=total_params, strategy=strategy)
    client_ids = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
    model_bytes = total_params * 4
    history, t0 = [], time.time()

    for episode in range(1, episodes + 1):
        updates, sizes = [], []
        for i, cid in enumerate(client_ids):
            local_model = copy.deepcopy(global_model)
            sample = partitions_pool[cid].sample(frac=0.8 if ordered else 1.0,
                                                 replace=not ordered,
                                                 random_state=seed * 1000 + episode * 10 + i)
            w_flat, n_used = client_update(local_model, sample, local_epochs, lr, batch_size,
                                           mode, window, dedup)
            updates.append(w_flat); sizes.append(n_used)

        new_flat = server.aggregate(global_model, updates, sizes)
        unpack_weights(global_model, new_flat)

        m = evaluate_global_model(global_model, eval_set, mode, window)
        m.update(model=model_type, strategy=strategy, episode=episode, params=total_params,
                 comm_MB=episode * 2 * len(client_ids) * model_bytes / 1e6,
                 time_s=time.time() - t0)
        history.append(m)
        print(f" -> Ep {episode} | Acc {m['acc']:.4f} | F1 {m['f1']:.4f} | Prec {m['prec']:.4f} | "
              f"Rec {m['rec']:.4f} | AUC {m['auc']:.4f} | FPR {m['fpr']:.4f} | {m['time_s']:.0f}s")
    return pd.DataFrame(history)

In [ ]:
test = execute_federated_experiment(clean_partitions, df_eval_clean, "GRU", "FedAvg",
                                    local_epochs=2, episodes=2)
test


[GRU] + [FedAvg] | epochs=2 | mode=window | dedup=True | window=10
 -> Ep 1 | Acc 0.7108 | F1 0.8027 | Prec 0.6929 | Rec 0.9539 | AUC 0.7080 | FPR 0.6808 | 70s
 -> Ep 2 | Acc 0.5886 | F1 0.6513 | Prec 0.6825 | Rec 0.6228 | AUC 0.7462 | FPR 0.4666 | 128s


,acc,f1,prec,rec,auc,fpr,model,strategy,episode,params,comm_MB,time_s
0,0.710759,0.802728,0.692929,0.953877,0.708034,0.680794,GRU,FedAvg,1,5057,0.242736,69.879411
1,0.588583,0.651322,0.682525,0.622847,0.746183,0.466602,GRU,FedAvg,2,5057,0.485472,128.264490


In [ ]:
results = []

In [ ]:
for st in ["FedAvg", "FedAdam", "FedYogi"]:
    results.append(execute_federated_experiment(clean_partitions, df_eval_clean, "FcNN", st))


[FcNN] + [FedAvg] | epochs=5 | mode=tabular | dedup=False
 -> Ep 1 | Acc 0.3832 | F1 0.0000 | Prec 0.0000 | Rec 0.0000 | AUC 0.6626 | FPR 0.0000 | 109s
 -> Ep 2 | Acc 0.5607 | F1 0.4960 | Prec 0.8485 | Rec 0.3504 | AUC 0.6958 | FPR 0.1007 | 215s
 -> Ep 3 | Acc 0.7522 | F1 0.8130 | Prec 0.7605 | Rec 0.8734 | AUC 0.8043 | FPR 0.4428 | 320s
 -> Ep 4 | Acc 0.7848 | F1 0.8179 | Prec 0.8553 | Rec 0.7836 | AUC 0.8579 | FPR 0.2134 | 427s
 -> Ep 5 | Acc 0.7779 | F1 0.8096 | Prec 0.8589 | Rec 0.7657 | AUC 0.8235 | FPR 0.2026 | 536s

[FcNN] + [FedAdam] | epochs=5 | mode=tabular | dedup=False
 -> Ep 1 | Acc 0.3832 | F1 0.0000 | Prec 0.0000 | Rec 0.0000 | AUC 0.6329 | FPR 0.0000 | 108s
 -> Ep 2 | Acc 0.6290 | F1 0.7143 | Prec 0.6802 | Rec 0.7520 | AUC 0.5621 | FPR 0.5690 | 220s
 -> Ep 3 | Acc 0.3832 | F1 0.0000 | Prec 0.0000 | Rec 0.0000 | AUC 0.5000 | FPR 0.0000 | 333s
 -> Ep 4 | Acc 0.3832 | F1 0.0000 | Prec 0.0000 | Rec 0.0000 | AUC 0.4999 | FPR 0.0000 | 443s
 -> Ep 5 | Acc 0.3832 | F1 0.0000 |

In [ ]:
for st in ["FedAvg", "FedAdam", "FedYogi"]:
    results.append(execute_federated_experiment(clean_partitions, df_eval_clean, "OneD_CNN", st))


[OneD_CNN] + [FedAvg] | epochs=5 | mode=tabular | dedup=False
 -> Ep 1 | Acc 0.6052 | F1 0.6604 | Prec 0.7032 | Rec 0.6226 | AUC 0.7390 | FPR 0.4229 | 92s
 -> Ep 2 | Acc 0.6656 | F1 0.6952 | Prec 0.7940 | Rec 0.6182 | AUC 0.8022 | FPR 0.2582 | 183s
 -> Ep 3 | Acc 0.8721 | F1 0.8878 | Prec 0.9680 | Rec 0.8198 | AUC 0.9020 | FPR 0.0437 | 274s
 -> Ep 4 | Acc 0.7287 | F1 0.7247 | Prec 0.9685 | Rec 0.5789 | AUC 0.9183 | FPR 0.0303 | 366s
 -> Ep 5 | Acc 0.7707 | F1 0.7777 | Prec 0.9674 | Rec 0.6502 | AUC 0.9571 | FPR 0.0353 | 455s

[OneD_CNN] + [FedAdam] | epochs=5 | mode=tabular | dedup=False
 -> Ep 1 | Acc 0.6268 | F1 0.7677 | Prec 0.6230 | Rec 1.0000 | AUC 0.5978 | FPR 0.9739 | 94s
 -> Ep 2 | Acc 0.6388 | F1 0.7735 | Prec 0.6307 | Rec 0.9998 | AUC 0.6116 | FPR 0.9422 | 193s
 -> Ep 3 | Acc 0.5714 | F1 0.6584 | Prec 0.6475 | Rec 0.6696 | AUC 0.6605 | FPR 0.5868 | 285s
 -> Ep 4 | Acc 0.4363 | F1 0.1592 | Prec 0.9954 | Rec 0.0865 | AUC 0.7007 | FPR 0.0007 | 377s
 -> Ep 5 | Acc 0.3833 | F1 0.

In [ ]:
for st in ["FedAvg", "FedAdam", "FedYogi"]:
    results.append(execute_federated_experiment(clean_partitions, df_eval_clean, "GRU", st, window=10))


[GRU] + [FedAvg] | epochs=5 | mode=window | dedup=True | window=10
 -> Ep 1 | Acc 0.6869 | F1 0.7771 | Prec 0.6929 | Rec 0.8845 | AUC 0.7161 | FPR 0.6313 | 127s
 -> Ep 2 | Acc 0.6027 | F1 0.6630 | Prec 0.6955 | Rec 0.6334 | AUC 0.7029 | FPR 0.4467 | 238s
 -> Ep 3 | Acc 0.5685 | F1 0.6206 | Prec 0.6782 | Rec 0.5721 | AUC 0.6658 | FPR 0.4372 | 346s
 -> Ep 4 | Acc 0.5603 | F1 0.6126 | Prec 0.6711 | Rec 0.5635 | AUC 0.6434 | FPR 0.4448 | 457s
 -> Ep 5 | Acc 0.5536 | F1 0.6031 | Prec 0.6680 | Rec 0.5498 | AUC 0.6422 | FPR 0.4401 | 564s

[GRU] + [FedAdam] | epochs=5 | mode=window | dedup=True | window=10
 -> Ep 1 | Acc 0.7034 | F1 0.8062 | Prec 0.6753 | Rec 1.0000 | AUC 0.6762 | FPR 0.7744 | 111s
 -> Ep 2 | Acc 0.6029 | F1 0.6180 | Prec 0.7600 | Rec 0.5207 | AUC 0.6306 | FPR 0.2648 | 220s
 -> Ep 3 | Acc 0.4999 | F1 0.5599 | Prec 0.6124 | Rec 0.5157 | AUC 0.6310 | FPR 0.5257 | 327s
 -> Ep 4 | Acc 0.4893 | F1 0.5714 | Prec 0.5924 | Rec 0.5517 | AUC 0.5126 | FPR 0.6113 | 436s
 -> Ep 5 | Acc 0.

In [ ]:
for st in ["FedAvg", "FedAdam", "FedYogi"]:
    results.append(execute_federated_experiment(clean_partitions, df_eval_clean, "LSTM", st, window=10))


[LSTM] + [FedAvg] | epochs=5 | mode=window | dedup=True | window=10
 -> Ep 1 | Acc 0.7969 | F1 0.8428 | Prec 0.8064 | Rec 0.8827 | AUC 0.8560 | FPR 0.3413 | 88s
 -> Ep 2 | Acc 0.7001 | F1 0.7413 | Prec 0.7923 | Rec 0.6965 | AUC 0.7727 | FPR 0.2940 | 173s
 -> Ep 3 | Acc 0.6710 | F1 0.7106 | Prec 0.7768 | Rec 0.6548 | AUC 0.7326 | FPR 0.3029 | 256s
 -> Ep 4 | Acc 0.6433 | F1 0.6777 | Prec 0.7658 | Rec 0.6077 | AUC 0.7314 | FPR 0.2993 | 338s
 -> Ep 5 | Acc 0.6369 | F1 0.6693 | Prec 0.7638 | Rec 0.5956 | AUC 0.7355 | FPR 0.2965 | 421s

[LSTM] + [FedAdam] | epochs=5 | mode=window | dedup=True | window=10
 -> Ep 1 | Acc 0.5442 | F1 0.5423 | Prec 0.7127 | Rec 0.4376 | AUC 0.7363 | FPR 0.2842 | 83s
 -> Ep 2 | Acc 0.6096 | F1 0.6647 | Prec 0.7069 | Rec 0.6273 | AUC 0.6962 | FPR 0.4188 | 166s
 -> Ep 3 | Acc 0.3858 | F1 0.0093 | Prec 0.9587 | Rec 0.0047 | AUC 0.7503 | FPR 0.0003 | 252s
 -> Ep 4 | Acc 0.5296 | F1 0.5785 | Prec 0.6468 | Rec 0.5233 | AUC 0.5958 | FPR 0.4603 | 338s
 -> Ep 5 | Acc 0.